# [Week 1] MLOps의 첫 단추: Automated Machine Learning (AutoML)

**목표:** 복잡한 데이터 전처리 및 알고리즘 튜닝 과정 없이, AutoML 도구(AutoGluon)를 활용하여 캘리포니아 부동산 가격을 예측하는 SOTA(State-of-the-Art) 베이스라인 모델을 구축합니다.

## 왜 AutoML인가요?
AI 엔지니어의 핵심 가치는 '반복적인 튜닝'이 아니라 '비즈니스 문제 정의'와 '양질의 데이터 확보'에 있습니다.  
우리는 오늘 결측치 처리, 범주형 변수 인코딩, 스케일링, 하이퍼파라미터 서치 과정을 단 3줄의 코드로 추상화하는 “AutoML의 마법”을 경험할 것입니다.

> 💡 **사용 라이브러리:** `AutoGluon` (AWS에서 개발한 오픈소스 AutoML 프레임워크로, 텍스트/이미지/정형 데이터에 모두 강력한 앙상블 성능을 보여줍니다.)


> **🗣️ [발표자 스크립트]**
> "자, 여러분. 앞서 이론 시간에서 MLOps의 방대한 파이프라인을 보셨죠?  
> 막상 현업에 가서 '당장 다음 주까지 부동산 가격 예측 모델 하나 뽑아보세요'라는 미션을 받으면 눈앞이 캄캄해집니다.  
> 결측치 채우고, 스케일링하고, XGBoost 파라미터 깎다가 일주일이 다 가버리거든요.
>
> 오늘 우리는 그 지루한 '학습 파이프라인'을 극도로 추상화해 주는 **AutoGluon**이라는 무기를 써볼 겁니다.  
> 구글 코랩 환경에서 아래 셀을 실행해 AutoGluon을 설치해 주세요. 약 1~2분 정도 소요됩니다."


## UV는 python을 버전별로 설치하고 환경변수 세팅을 하는데 있어서, virtualenv, pyenv보다 더 최근에 핫한 프로그램 입니다.
## pip, pip3가 안 먹힐 수 있지만 해당 프로그램을 활용해보시기 바랍니다.

In [10]:
# # 1. uv 설치
# curl -LsSf https://astral.sh/uv/install.sh | sh

# # 2. Python 여러 버전 설치 필요한 경우 설치
# uv python install 3.10 3.11 3.12 3.13

# # 3. 프로젝트 생성
# mkdir mlops-project
# cd mlops-project

# # 4. 특정 버전 고정
# uv python pin 3.13

# # 5. 가상환경 생성
# uv venv

# # 6. 패키지 설치
# uv add pandas scikit-learn autogluon

In [5]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [9]:
# AutoGluon 라이브러리 설치 (Colab 환경 기준)
# - 로컬 Jupyter에서도 동작하지만, 환경/OS에 따라 설치 시간이 달라질 수 있습니다.
!uv add pandas scikit-learn autogluon

import warnings
warnings.filterwarnings('ignore')  # 불필요한 경고 메시지 숨김

error: No `pyproject.toml` found in current directory or any parent directory


## 1. 데이터 로드 (Data Extraction)

실습에 사용할 데이터는 머신러닝의 고전인 **'캘리포니아 주택 가격 데이터셋(California Housing Dataset)'**입니다.

- **목적:** 해당 지역의 인구통계학적 정보와 위치 정보를 바탕으로 주택 가격의 중앙값(Target)을 예측합니다.
- 이는 실제 프롭테크(PropTech)나 금융권의 담보대출 한도 산정 모델에서 다루는 문제와 궤를 같이합니다.


> **🗣️ [발표자 스크립트]**
> "데이터를 불러오겠습니다. 사이킷런에서 제공하는 캘리포니아 주택 가격 데이터입니다.  
> 금융권에서 주택 담보대출 한도를 산정할 때, 이 집의 적정 가치가 얼마인지 예측하는 AVM(자동가치산정) 모델의 아주 기본적인 형태라고 보시면 됩니다.  
> 데이터를 판다스(Pandas) 데이터프레임으로 변환해서 눈으로 직접 확인해 보시죠."


In [8]:
import pandas as pd
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from autogluon.tabular import TabularDataset, TabularPredictor

# 1) 데이터 불러오기
housing = fetch_california_housing()

# 2) 특징(Feature)들을 DataFrame으로 변환
df = pd.DataFrame(housing.data, columns=housing.feature_names)

# 3) 예측해야 할 정답(Target) 컬럼 추가
#    - 캘리포니아 주택 데이터셋의 target은 보통 "중앙값 집값" (단위/스케일은 데이터셋 정의에 따름) 입니다.
label = 'Price'
df[label] = housing.target

print(f"데이터 형태: {df.shape}")
display(df.head())

# 4) Train / Test 분할 (8:2)
#    - '미래 데이터'를 가정한 테스트 셋으로 일반화 성능을 평가합니다.
train_data, test_data = train_test_split(df, test_size=0.2, random_state=42)

# 5) AutoGluon이 인식할 수 있는 전용 Dataset 형태로 변환
train_dataset = TabularDataset(train_data)
test_dataset  = TabularDataset(test_data)

print(f"Train: {train_dataset.shape} / Test: {test_dataset.shape}")


ModuleNotFoundError: No module named 'autogluon'

## 2. AutoML 모델 학습 (Model Training)

이제 AutoGluon을 활용해 예측 모델을 만듭니다. 우리는 그 어떤 변수 전처리 코드도 작성하지 않습니다.

- `label`: 우리가 예측하고자 하는 타겟 컬럼명입니다.
- `eval_metric`: 모델의 성능을 평가할 지표입니다. 회귀(Regression) 문제이므로 **RMSE(Root Mean Squared Error)**를 사용합니다.
- `time_limit`: AutoML이 탐색을 수행할 최대 시간(초)입니다.
- `presets`: 모델의 학습 전략입니다. `'best_quality'`는 가능한 모든 앙상블과 스태킹(Stacking) 기법을 동원하여 성능을 쥐어짜는 옵션입니다.


> **🗣️ [발표자 스크립트]**
> "가장 중요한 부분입니다. 화면을 잠깐 보실까요?  
> 일반적으로 머신러닝 코드를 짜면 StandardScaler로 스케일링하고, 카테고리 변수 One-hot 인코딩하고 코드가 길어지죠.  
> 하지만 코드를 보시면 그런 게 전혀 없습니다.
>
> 타겟 컬럼 이름만 지정해 주고 `TabularPredictor.fit()`을 호출합니다.  
> 실습 시간 관계상 `time_limit`은 120초(2분)로 주겠습니다.  
> 2분 동안 이 엔진이 내부적으로 LightGBM, XGBoost, Random Forest 등 수십 개의 모델을 만들고,  
> 그 모델들을 합쳐서 최적의 앙상블 모델까지 알아서 구축합니다. 실행해 보겠습니다."


In [ ]:
print("--- 🚀 AutoML Training Starts ---")

# 모델이 저장될 디렉토리 경로
save_path = 'ag_models/'

# ✅ 핵심: "단 2줄"로 전처리 + 모델 후보군 학습 + 앙상블까지 자동 수행
predictor = TabularPredictor(
    label=label,
    eval_metric='root_mean_squared_error',
    path=save_path
).fit(
    train_data=train_dataset,
    time_limit=120,          # 최대 2분간 최적의 모델 탐색
    presets='best_quality'   # 최상의 예측 성능을 위한 앙상블/스태킹 강화
)

print("--- 🏁 AutoML Training Completed ---")


Verbosity: 2 (Standard Logging)


--- 🚀 AutoML Training Starts ---


=================== System Info ===================
AutoGluon Version:  1.5.0
Python Version:     3.13.12
Operating System:   Darwin
Platform Machine:   arm64
Platform Version:   Darwin Kernel Version 25.3.0: Wed Jan 28 20:48:41 PST 2026; root:xnu-12377.81.4~5/RELEASE_ARM64_T6041
CPU Count:          14
Pytorch Version:    2.9.1
CUDA Version:       CUDA is not available
GPU Count:          WARNING: Exception was raised when calculating GPU count (AssertionError)
Memory Avail:       5.23 GB / 24.00 GB (21.8%)
Disk Space Avail:   700.58 GB / 926.35 GB (75.6%)
Presets specified: ['best_quality']
Using hyperparameters preset: hyperparameters='zeroshot'
Setting dynamic_stacking from 'auto' to True. Reason: Enable dynamic_stacking when use_bag_holdout is disabled. (use_bag_holdout=False)
Stack configuration (auto_stack=True): num_stack_levels=1, num_bag_folds=8, num_bag_sets=1
DyStack is enabled (dynamic_stacking=True). AutoGluon will try to determine whether the input data is affected by sta

--- 🏁 AutoML Training Completed ---


## 3. 리더보드 (Leaderboard) 분석

AutoML은 단 하나의 모델만 만드는 것이 아니라, 수많은 알고리즘을 테스트하고 경쟁시킵니다.  
어떤 모델이 가장 우수한 성능을 냈는지 `leaderboard`를 통해 확인합니다.


> **🗣️ [발표자 스크립트]**
> "학습이 끝났습니다. Kaggle 같은 데이터 분석 대회에 나가면 리더보드(순위표)가 있죠?  
> AutoGluon도 스스로 경쟁을 시킨 결과를 리더보드로 보여줍니다.  
> 결과를 보시면 `WeightedEnsemble_L2` 혹은 `L3` 같은 모델이 1위에 있을 겁니다.
>
> 단일 알고리즘(예: XGBoost 하나)을 쓰는 것보다, 여러 모델의 장점만 취합한 앙상블 모델이  
> 실무에서도 훨씬 강력하다는 것을 기계가 스스로 증명해 주는 겁니다."


In [ ]:
# Test 데이터셋을 기반으로 생성된 모델들의 순위표 출력
leaderboard = predictor.leaderboard(test_dataset, silent=True)

display(leaderboard.head(10))  # 상위 10개 모델 확인


,model,score_test,score_val,eval_metric,pred_time_test,pred_time_val,fit_time,pred_time_test_marginal,pred_time_val_marginal,fit_time_marginal,stack_level,can_infer,fit_order
0,NeuralNetFastAI_BAG_L2,-0.417988,-0.430721,root_mean_squared_error,1.056261,1.011328,64.427952,0.120792,0.061045,5.050756,2,True,9
1,WeightedEnsemble_L3,-0.418346,-0.428900,root_mean_squared_error,1.288836,1.882582,70.799145,0.000920,0.000197,0.008995,3,True,10
2,CatBoost_BAG_L1,-0.422332,-0.435451,root_mean_squared_error,0.139478,0.080951,35.110947,0.139478,0.080951,35.110947,1,True,2
3,CatBoost_BAG_L2,-0.423872,-0.433002,root_mean_squared_error,0.946861,0.958999,60.703448,0.011392,0.008716,1.326252,2,True,7
4,WeightedEnsemble_L2,-0.423975,-0.434346,root_mean_squared_error,0.337647,0.483991,37.476473,0.000820,0.000701,0.030471,2,True,5
5,ExtraTreesMSE_BAG_L2,-0.425412,-0.436221,root_mean_squared_error,1.043712,1.371198,60.220975,0.108243,0.420915,0.843779,2,True,8
6,RandomForestMSE_BAG_L2,-0.431255,-0.444178,root_mean_squared_error,1.047489,1.391709,63.569363,0.112020,0.441426,4.192167,2,True,6
7,RandomForestMSE_BAG_L1,-0.501311,-0.501400,root_mean_squared_error,0.197349,0.402339,2.335055,0.197349,0.402339,2.335055,1,True,1
8,ExtraTreesMSE_BAG_L1,-0.511131,-0.506013,root_mean_squared_error,0.120009,0.407548,0.597830,0.120009,0.407548,0.597830,1,True,3
9,NeuralNetFastAI_BAG_L1,-0.555443,-0.567663,root_mean_squared_error,0.478633,0.059445,21.333364,0.478633,0.059445,21.333364,1,True,4


## 4. 예측 및 성능 평가 (Evaluation & Inference)

이제 학습 과정에서 전혀 보지 못한 `test_dataset`을 모델에 통과시켜 실제 예측값(Prediction)을 얻고,  
실제 정답(Ground Truth)과 비교해 봅니다.


> **🗣️ [발표자 스크립트]**
> "이제 성적표를 받아볼 시간입니다.  
> 분리해 두었던 테스트 데이터를 모델에 밀어 넣어서 집값을 예측해 보겠습니다.  
> 눈으로 직접 확인해 볼까요? 실제 가격(Actual)과 우리 모델이 예측한 가격(Predicted)의 오차가 어느 정도인지 비교해 보십시오.
>
> 현업에서는 이 베이스라인 모델을 만드는데 걸린 시간을 획기적으로 줄인 덕분에,  
> 남는 시간에 '이 지역의 학군 데이터'나 '지하철역과의 거리' 같은 파생 변수를 추가로 수집하는 데 집중할 수 있게 됩니다.  
> 이것이 AI Bengineer가 일하는 방식입니다."


In [ ]:
# 모델 평가 (RMSE 오차 확인)
performance = predictor.evaluate(test_dataset)

print("✅ Test 데이터셋 최종 평가 결과:")
for metric, score in performance.items():
    print(f"- {metric}: {score:.4f}")

# Test 데이터셋 예측 수행 (Target 컬럼 제외하고 입력)
X_test = test_dataset.drop(columns=[label])
y_pred = predictor.predict(X_test)

# 실제 가격과 예측 가격을 비교하는 DataFrame 생성
comparison_df = pd.DataFrame({
    'Actual Price': test_dataset[label],
    'Predicted Price': y_pred
})

# 절대 오차(Absolute Error) 계산
comparison_df['Error'] = (comparison_df['Actual Price'] - comparison_df['Predicted Price']).abs()

print("\n🔍 실제 집값 vs 예측 집값 비교 (Top 10):")
display(comparison_df.head(10))


/Users/sungjae-cha/Documents/@03 아주대AI대학원/2026-1st-semester/lecture ppt/.venv/lib/python3.13/site-packages/scipy/stats/_stats_py.py:4776: RuntimeWarning: overflow encountered in vecdot
  r = xp.vecdot(xm / normxm, ym / normym, axis=axis)


✅ Test 데이터셋 최종 평가 결과:
- root_mean_squared_error: -0.4183
- mean_squared_error: -0.1750
- mean_absolute_error: -0.2683
- r2: 0.8664
- pearsonr: 0.9309
- median_absolute_error: -0.1667

🔍 실제 집값 vs 예측 집값 비교 (Top 10):


,Actual Price,Predicted Price,Error
20046,0.47700,0.516743,0.039743
3024,0.45800,0.752395,0.294395
15663,5.00001,5.016681,0.016671
20484,2.18600,2.437969,0.251969
9814,2.78000,2.595311,0.184689
13311,1.58700,1.611363,0.024363
7113,1.98200,2.295859,0.313859
7668,1.57500,1.550674,0.024326
18246,3.40000,2.992818,0.407182
5723,4.46600,4.960176,0.494176


## 💡 Wrap-up & Next Step

오늘 우리는 **코드 3줄**로 복잡한 데이터 전처리와 파라미터 튜닝을 우회하여 뛰어난 성능의 베이스라인 모델을 구축했습니다.

### [생각해 볼 문제 - Next MLOps Step]
1. 이 훌륭한 모델 파일(Artifacts)을 우리 회사 운영 서버에 올리려면 어떻게 해야 할까요?
2. 만약 내일 캘리포니아에 엄청난 지진이 나서 집값이 폭락한다면(Data Drift), 이 모델의 성능은 어떻게 될까요? 어떻게 자동으로 재학습 시킬 수 있을까요?

➡️ **다음 주차(Week 2~3) 예고:** 블랙박스처럼 지나간 **데이터 전처리**의 원리를 직접 파헤쳐보고, 이 코드를 형상 관리하기 위한 **Git과 DVC(Data Version Control)** 실습으로 넘어갑니다.


> **🗣️ [발표자 스크립트]**
> "여러분, 오늘 실습 어떠셨나요? 너무 쉬워서 놀라셨을 수도 있습니다. 하지만 진짜 고민은 지금부터입니다.
>
> 이 좋은 모델을 만들었는데, 회사 웹사이트에 연동하려면 어떻게 해야 할까요?  
> 또, 만약 내일 경제 위기가 와서 캘리포니아 집값이 반토막이 나면 우리 모델은 바보가 될 텐데,  
> 언제 어떻게 다시 학습을 시킬까요?
>
> 이 질문들에 대한 답을 찾아가는 과정이 바로 앞으로 15주 동안 우리가 배울 MLOps입니다.
>
> 다음 시간에는 모델링 이전 단계로 돌아가, 데이터를 뜯어보고 버전을 관리하는 인프라 기초부터 다져보겠습니다.  
> 오늘 수고 많으셨습니다!"
